In [12]:
pip install sktime scikit-learn numpy pandas

In [13]:
import numpy as np
import pandas as pd
from sktime.transformations.panel.rocket import Rocket
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelBinarizer

In [15]:
import pandas as pd
import numpy as np


file_path = "trainDataset_w_aug.csv"
trainDataset = pd.read_csv(file_path, index_col=[0, 1])

file_path = "testDataset_w_aug.csv"
testDataset = pd.read_csv(file_path, index_col=[0, 1])


In [16]:
import pandas as pd

# Define the original pattern mapping
pattern_encoding = {
    'Double Top, Adam and Adam': 0,
    'Triangle, symmetrical': 1,
    'Double Bottom, Eve and Adam': 2,
    'Head-and-shoulders top': 3,
    'Double Bottom, Adam and Adam': 4,
    'Head-and-shoulders bottom': 5,
    'Flag, high and tight': 6,
    'Cup with handle': 7,
    'No Pattern': 8
}

# Reverse mapping (for filtering)
pattern_decoding = {v: k for k, v in pattern_encoding.items()}

# Load dataset (replace with actual loading method)
# trainDataset = pd.read_csv('train.csv')
# testDataset = pd.read_csv('test.csv')

# Step 1: Drop "Cup with Handle" (Pattern 7)
trainDataset = trainDataset[trainDataset['Pattern'] != pattern_encoding['Cup with handle']]
testDataset = testDataset[testDataset['Pattern'] != pattern_encoding['Cup with handle']]

# Step 2: Merge "Double Bottom" classes (2 → 4)
trainDataset['Pattern'] = trainDataset['Pattern'].replace({2: 4})
testDataset['Pattern'] = testDataset['Pattern'].replace({2: 4})

# Step 3: Ensure sequential labels for XGBoost
unique_patterns = sorted(trainDataset['Pattern'].unique())  # Get unique remaining classes
new_pattern_mapping = {old: new for new, old in enumerate(unique_patterns)}

# Apply mapping
trainDataset['Pattern'] = trainDataset['Pattern'].map(new_pattern_mapping)
testDataset['Pattern'] = testDataset['Pattern'].map(new_pattern_mapping)

# Step 4: Verify results
print("Updated Class Labels Mapping:", new_pattern_mapping)
print("Updated Training Class Distribution:\n", trainDataset['Pattern'].value_counts())
print("Updated Testing Class Distribution:\n", testDataset['Pattern'].value_counts())

# Your dataset is now ready for training!


Updated Class Labels Mapping: {np.int64(0): 0, np.int64(1): 1, np.int64(3): 2, np.int64(4): 3, np.int64(5): 4, np.int64(6): 5}
Updated Training Class Distribution:
 Pattern
1    34557
5    34155
3    27553
4    26990
2    24945
0    13368
Name: count, dtype: int64
Updated Testing Class Distribution:
 Pattern
1    4238
3    3395
2    2557
4    2048
5    1764
0    1421
Name: count, dtype: int64


<ipython-input-16-7446746d3a93>:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  trainDataset['Pattern'] = trainDataset['Pattern'].replace({2: 4})
<ipython-input-16-7446746d3a93>:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  testDataset['Pattern'] = testDataset['Pattern'].replace({2: 4})
<ipython-input-16-7446746d3a93>:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentat

In [18]:
import numpy as np
import pandas as pd

def adjust_series_length(group, target_length):

    series = group.values
    current_length = len(series)

    if current_length > target_length:
        return series[:target_length]
    else:
        # Padding with zeros if shorter
        padding = np.zeros((target_length - current_length, series.shape[1]))
        return np.vstack([series, padding])

In [20]:
features = ['Open', 'High', 'Low', 'Close', 'Volume']
target = 'Pattern'
series_length = 275  # Target series length for ROCKET

def prepare_rocket_data(dataset, features, target, series_length):
    adjusted = dataset.groupby(level=0).apply(
        lambda group: adjust_series_length(group[features], series_length)
    )

    X = np.stack(adjusted.values)


    y = dataset.groupby(level=0)[target].first().values

    return X, y


X_train, y_train = prepare_rocket_data(trainDataset, features, target, series_length)
X_test, y_test = prepare_rocket_data(testDataset, features, target, series_length)

X_train = np.transpose(X_train, (0, 2, 1))
X_test = np.transpose(X_test, (0, 2, 1))

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

X_train shape: (6480, 5, 275)
y_train shape: (6480,)


In [21]:
from xgboost import XGBClassifier
from sktime.transformations.panel.rocket import Rocket
from sklearn.pipeline import make_pipeline

rocket = Rocket(num_kernels=7000)

xgb_clf = XGBClassifier(
    use_label_encoder=False,
    eval_metric='mlogloss',
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    min_child_weight=7,
    gamma=0.3,
    subsample=0.7, colsample_bytree=0.7,
    reg_alpha=0.5, reg_lambda=1.5
)

# Create and fit the pipeline
clf_xgb = make_pipeline(rocket, xgb_clf)
clf_xgb.fit(X_train, y_train)

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [11:33:55] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Pipeline(steps=[('rocket', Rocket(num_kernels=7000)),
                ('xgbclassifier',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.7, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric='mlogloss',
                               feature_types=None, gamma=0.3, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=3, max_leaves=None, min_child_weight=7,
                               missing=nan, monotone_constraints=None,
                               multi_strategy=None, n_estimators=300,
                               n_jobs=None, num_parallel_tree=None,
                               objective='multi:softprob', ...))])

In [22]:
y_train_probs = clf_xgb.predict_proba(X_train)
y_test_probs = clf_xgb.predict_proba(X_test)

In [23]:
from sklearn.metrics import accuracy_score

# Training accuracy
y_train_pred = y_train_probs.argmax(axis=1)
train_accuracy = accuracy_score(y_train, y_train_pred)
print(f"Training Accuracy: {train_accuracy:.2f}")

# Testing accuracy
y_test_pred = y_test_probs.argmax(axis=1)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {test_accuracy:.2f}")

Training Accuracy: 0.97
Test Accuracy: 0.83
